In [1]:
import pandas as pd
import numpy as np
import psychrolib
import CoolProp.CoolProp as CP

# Initialize psychrolib in SI units
psychrolib.SetUnitSystem(psychrolib.SI)

# --- CONFIGURATION & CONSTANTS (From PDF Architecture) ---
IT_LOAD_MW = 10.0
IT_LOAD_KW = IT_LOAD_MW * 1000.0
HOURS_PER_YEAR = 8760
WUE_BASELINE = 1.8  # L/kWh
RO_RECOVERY_RATE = 0.67
M_AIR = 50.0  # Air mass flow rate (kg/s)
CP_AIR = 1.005  # kJ/kg.C
T_EXHAUST = 55.0  # Celsius

# Financial Constants
CAPEX_TRADITIONAL = 10000000
CAPEX_AWG = CAPEX_TRADITIONAL + 4500000
CAPEX_SCO2 = 30000000
OPEX_TRADITIONAL_BASE = 2000000
DISCOUNT_RATE = 0.07
YEARS = 10

def load_epw_weather(file_path):
    """
    Parses a standard EPW file and extracts Temperature and Relative Humidity for 8760 hours.
    If your file is already a clean CSV, you can use pd.read_csv(file_path) directly.
    """
    try:
        # EPW data typically starts from line 9 (0-indexed line 8)
        df = pd.read_csv(file_path, skiprows=8, header=None)
        
        # Standard EPW columns: Column 6 is Dry Bulb Temperature, Column 8 is Relative Humidity
        weather_df = pd.DataFrame({
            'Temperature': df[6].values,
            'Humidity': df[8].values
        })
        
        # Ensure exactly 8760 hours
        if len(weather_df) != HOURS_PER_YEAR:
            raise ValueError(f"File does not contain exactly 8760 hours of data. Found: {len(weather_df)}")
        return weather_df
    except Exception as e:
        print(f"Error reading file {file_path}. Generating realistic mock 8760-hour cycle for fallback.")
        # Fallback realistic data generator if file path is wrong or corrupted
        np.random.seed(42)
        mock_temp = 25.0 + 10.0 * np.sin(np.linspace(0, 2 * np.pi * 365, HOURS_PER_YEAR)) + np.random.normal(0, 2, HOURS_PER_YEAR)
        mock_hum = 70.0 - 20.0 * np.sin(np.linspace(0, 2 * np.pi * 365, HOURS_PER_YEAR)) + np.random.normal(0, 5, HOURS_PER_YEAR)
        mock_hum = np.clip(mock_hum, 10, 100)
        return pd.DataFrame({'Temperature': mock_temp, 'Humidity': mock_hum})

def run_yearly_simulation(weather_df, location_name):
    print(f"\nRunning 8,760-hour simulation for: {location_name}...")
    
    total_baseline_water_liters = 0.0
    total_extracted_groundwater_liters = 0.0
    total_awg_water_generated_liters = 0.0
    
    # sCO2 constant properties over the loop to save computational time
    try:
        sco2_density = CP.PropsSI('D', 'P', 10e6, 'T', 35 + 273.15, 'CO2')
        sco2_cp = CP.PropsSI('C', 'P', 10e6, 'T', 35 + 273.15, 'CO2')
    except:
        sco2_density, sco2_cp = 700.0, 2000.0  # Fallback analytical constants if CoolProp fails
        
    # Microchannel pressure drop calculation
    delta_p = 0.02 * (0.05 / 0.001) * (sco2_density * 2.0**2 / 2)
    compressor_power_w = (delta_p * 0.005) / 0.70
    annual_sco2_energy_kwh = (compressor_power_w / 1000.0) * HOURS_PER_YEAR

    # Hourly Simulation Loop (Iterating over 8,760 rows)
    for hour in range(HOURS_PER_YEAR):
        t_ambient = weather_df.loc[hour, 'Temperature']
        rh_percent = weather_df.loc[hour, 'Humidity']
        rh_fraction = rh_percent / 100.0
        
        # 1. Baseline Calculations (Hourly)
        hourly_it_energy_kwh = IT_LOAD_KW * 1.0  # 1 hour duration
        hourly_baseline_water = hourly_it_energy_kwh * WUE_BASELINE
        hourly_groundwater = hourly_baseline_water / RO_RECOVERY_RATE
        
        total_baseline_water_liters += hourly_baseline_water
        total_extracted_groundwater_liters += hourly_groundwater
        
        # 2. AWG Calculations (Hourly)
        try:
            # Get Humidity Ratio (kg water / kg dry air)
            humidity_ratio = psychrolib.GetHumRatioFromRelHum(t_ambient, rh_fraction, 101325)
        except:
            humidity_ratio = 0.015  # Default safety fallback
            
        q_waste_kw = M_AIR * CP_AIR * (T_EXHAUST - t_ambient)
        
        moisture_capture_limit = humidity_ratio * M_AIR * 3600.0  # Convert kg/s to Liters/hr (approx 1kg=1L)
        heat_regeneration_limit = max(0.0, q_waste_kw * 0.5)
        condenser_limit = 2000.0
        
        hourly_awg_water = min(moisture_capture_limit, heat_regeneration_limit, condenser_limit)
        total_awg_water_generated_liters += hourly_awg_water

    # 3. Post-Loop Aggregations & Solutions Comparison
    net_awg_groundwater_withdrawal = max(0.0, total_extracted_groundwater_liters - total_awg_water_generated_liters)
    wue_awg_net = (total_baseline_water_liters - total_awg_water_generated_liters) / (IT_LOAD_KW * HOURS_PER_YEAR)
    
    # 4. Societal Impact Model (Experiment 7)
    # Average family water consumption parameter = 91,250 Liters/year
    water_saved_liters = total_extracted_groundwater_liters - net_awg_groundwater_withdrawal
    households_sustained = water_saved_liters / 91250.0
    
    # 5. Financial Evaluation (10-Year NPV)
    annual_savings_awg = (total_extracted_groundwater_liters - net_awg_groundwater_withdrawal) * 0.002  # $0.002 per Liter
    additional_capex_awg = CAPEX_AWG - CAPEX_TRADITIONAL
    npv_awg = sum(annual_savings_awg / ((1 + DISCOUNT_RATE) ** t) for t in range(1, YEARS + 1)) - additional_capex_awg
    
    # Print Comprehensive Academic Summary Tables as requested by Supervisor
    print(f"=== SIMULATION SUMMARY FOR {location_name.upper()} ===")
    print(f"{"Metric":<45} | {"Value":<15}")
    print("-" * 65)
    print(f"{"Total Annual IT Energy Consumption":<45} | {IT_LOAD_KW * HOURS_PER_YEAR:,.0f} kWh")
    print(f"{"Baseline Annual Water Consumption":<45} | {total_baseline_water_liters:,.2f} Liters")
    print(f"{"Baseline Groundwater Extraction (with RO)":<45} | {total_extracted_groundwater_liters:,.2f} Liters")
    print(f"{"AWG Total Annual Water Generated":<45} | {total_awg_water_generated_liters:,.2f} Liters")
    print(f"{"Net AWG Groundwater Extraction":<45} | {net_awg_groundwater_withdrawal:,.2f} Liters")
    print(f"{"Net AWG System WUE":<45} | {wue_awg_net:.4f} L/kWh")
    print(f"{"sCO2 Closed-Loop System WUE":<45} | {0.0000:.4f} L/kWh")
    print(f"{"sCO2 Annual Parasitic Energy Overhead":<45} | {annual_sco2_energy_kwh:,.2f} kWh")
    print(f"{"Social Benefit: Equivalent Households Sustained":<45} | {households_sustained:,.1f} Families")
    print(f"{"10-Year Financial NPV for AWG Integration":<45} | ${npv_awg:,.2f}")
    print("=" * 65)

# --- EXECUTION PIPELINE ---
if __name__ == "__main__":
    # Change these paths to your actual downloaded .epw or .csv files
    dhaka_weather_path = "Dhaka_Weather.epw"
    dubai_weather_path = "Dubai_Weather.epw"
    london_weather_path = "London_Weather.epw"
    
    # Load and execute simulations
    dhaka_data = load_epw_weather(dhaka_weather_path)
    run_yearly_simulation(dhaka_data, "Dhaka (Humid Climate)")
    
    dubai_data = load_epw_weather(dubai_weather_path)
    run_yearly_simulation(dubai_data, "Dubai (Arid Climate)")
    
    london_data = load_epw_weather(london_weather_path)
    run_yearly_simulation(london_data, "London (Temperate Climate)")

Error reading file Dhaka_Weather.epw. Generating realistic mock 8760-hour cycle for fallback.

Running 8,760-hour simulation for: Dhaka (Humid Climate)...
=== SIMULATION SUMMARY FOR DHAKA (HUMID CLIMATE) ===
Metric                                        | Value          
-----------------------------------------------------------------
Total Annual IT Energy Consumption            | 87,600,000 kWh
Baseline Annual Water Consumption             | 157,680,000.00 Liters
Baseline Groundwater Extraction (with RO)     | 235,343,283.58 Liters
AWG Total Annual Water Generated              | 6,602,815.08 Liters
Net AWG Groundwater Extraction                | 228,740,468.50 Liters
Net AWG System WUE                            | 1.7246 L/kWh
sCO2 Closed-Loop System WUE                   | 0.0000 L/kWh
sCO2 Annual Parasitic Energy Overhead         | 89.20 kWh
Social Benefit: Equivalent Households Sustained | 72.4 Families
10-Year Financial NPV for AWG Integration     | $-4,407,249.18
Error reading 